# Phase 5 — Drift Detection on Sensor Stream Anomaly Scores

Dự án: **An Evolving Fuzzy Reasoning System for Sensor Stream Anomaly Detection under Concept Drift**

### Pipeline:
```text
Sensor Stream → Static Fuzzy → Anomaly Score A(t) → Drift Detector → Drift Signal → Evolving Fuzzy → Adaptation
```

### Các giai đoạn chính trong Phase 5:
- **5.1** Chọn tín hiệu đưa vào detector & chuẩn bị anomaly-score stream
- **5.2** Thiết kế & xây dựng Drift Detector
- **5.3** Thực nghiệm trên kịch bản Control (No Injected Drift)
- **5.4** Thực nghiệm trên kịch bản Sudden Drift
- **5.5** Thực nghiệm trên kịch bản Gradual Drift
- **5.6** Đánh giá toàn diện detection behavior


In [1]:
# Phase 5.0 — Setup môi trường, Static Mamdani FIS Engine và nạp 3 kịch bản Stream
from pathlib import Path
import pandas as pd
import numpy as np
import skfuzzy as fuzz

# 1. Nạp dữ liệu gốc và trích xuất Test stream (Phase 1.5)
data_path = Path("../data/ai4i2020.csv") if Path("../data/ai4i2020.csv").exists() else Path("data/ai4i2020.csv")
df = pd.read_csv(data_path)
test_df = df.iloc[8000:].copy()
stream_df = test_df.copy().reset_index(drop=True)

# 2. Universe of Discourse (Phase 2 & Phase 4)
universes = {
    "air_temp":     np.linspace(295.3, 304.5, 1000),
    "process_temp": np.linspace(305.7, 313.8, 1000),
    "rpm":          np.linspace(1168,  2886,  1000),
    "torque":       np.linspace(3.8,   76.2,  1000),
    "tool_wear":    np.linspace(0,     253,   1000),
    "anomaly":      np.linspace(0,     1,     1000),
}

# 3. Cấu hình Static Mamdani FIS đã đóng băng (Phase 2 & Phase 3)
mf_params = {
    "air_temp": {
        "LOW":    ("trap", [295.3, 295.3, 298.0, 300.4]),
        "MEDIUM": ("tri",  [298.0, 300.4, 302.5]),
        "HIGH":   ("trap", [300.4, 302.5, 304.5, 304.5])
    },
    "process_temp": {
        "LOW":    ("trap", [305.7, 305.7, 308.0, 309.7]),
        "MEDIUM": ("tri",  [308.0, 309.7, 311.5]),
        "HIGH":   ("trap", [309.7, 311.5, 313.8, 313.8])
    },
    "rpm": {
        "LOW":    ("trap", [1168.0, 1168.0, 1350.0, 1500.0]),
        "MEDIUM": ("tri",  [1350.0, 1504.0, 1800.0]),
        "HIGH":   ("trap", [1600.0, 1880.0, 2886.0, 2886.0])
    },
    "torque": {
        "LOW":    ("trap", [3.8, 3.8, 25.0, 40.0]),
        "MEDIUM": ("tri",  [25.0, 40.0, 55.0]),
        "HIGH":   ("trap", [40.0, 55.0, 76.2, 76.2])
    },
    "tool_wear": {
        "LOW":    ("trap", [0.0, 0.0, 54.0, 109.0]),
        "MEDIUM": ("tri",  [54.0, 109.0, 164.0]),
        "HIGH":   ("trap", [109.0, 164.0, 253.0, 253.0])
    }
}

anomaly_universe = universes["anomaly"]
anomaly_mfs = {
    "LOW":    fuzz.trapmf(anomaly_universe, [0.0, 0.0, 0.25, 0.50]),
    "MEDIUM": fuzz.trimf( anomaly_universe, [0.25, 0.50, 0.75]),
    "HIGH":   fuzz.trapmf(anomaly_universe, [0.50, 0.75, 1.0, 1.0])
}

def eval_mf(x, mf_type, params):
    x = float(x)
    if mf_type == "tri":
        a, b, c = params
        if a < b and a <= x <= b:
            return (x - a) / (b - a)
        elif b < c and b <= x <= c:
            return (c - x) / (c - b)
        elif x == b:
            return 1.0
        return 0.0
    elif mf_type == "trap":
        a, b, c, d = params
        if x < a:
            return 1.0 if a == b else 0.0
        elif a <= x < b:
            return (x - a) / (b - a) if b > a else 1.0
        elif b <= x <= c:
            return 1.0
        elif c < x <= d:
            return (d - x) / (d - c) if d > c else 1.0
        else:
            return 1.0 if c == d else 0.0

def compute_memberships(sample):
    mapping = {
        "air_temp":     sample["Air temperature [K]"],
        "process_temp": sample["Process temperature [K]"],
        "rpm":          sample["Rotational speed [rpm]"],
        "torque":       sample["Torque [Nm]"],
        "tool_wear":    sample["Tool wear [min]"],
    }
    m = {}
    for var_name, val in mapping.items():
        m[var_name] = {}
        for term, (m_type, params) in mf_params[var_name].items():
            m[var_name][term] = eval_mf(val, m_type, params)
    return m

rules = [
    # HIGH anomaly — 4 rules
    ("R1", [("rpm", "LOW"), ("torque", "HIGH"), ("air_temp", "HIGH")], "HIGH"),
    ("R2", [("torque", "HIGH"), ("air_temp", "HIGH"), ("tool_wear", "HIGH")], "HIGH"),
    ("R3", [("rpm", "LOW"), ("torque", "HIGH"), ("process_temp", "HIGH")], "HIGH"),
    ("R4", [("rpm", "LOW"), ("torque", "HIGH"), ("tool_wear", "HIGH")], "HIGH"),

    # MEDIUM anomaly — 5 rules
    ("R5", [("rpm", "LOW"), ("torque", "HIGH")], "MEDIUM"),
    ("R6", [("rpm", "LOW"), ("air_temp", "HIGH")], "MEDIUM"),
    ("R7", [("torque", "HIGH"), ("air_temp", "HIGH")], "MEDIUM"),
    ("R8", [("torque", "HIGH"), ("tool_wear", "HIGH")], "MEDIUM"),
    ("R9", [("torque", "HIGH"), ("process_temp", "HIGH")], "MEDIUM"),

    # LOW anomaly — 3 rules
    ("R10", [("rpm", "MEDIUM"), ("torque", "MEDIUM")], "LOW"),
    ("R11", [("rpm", "MEDIUM"), ("torque", "LOW")], "LOW"),
    ("R12", [("rpm", "HIGH"), ("torque", "LOW")], "LOW"),
]

def mamdani_inference(sample, return_details=False):
    mu = compute_memberships(sample)
    rule_activations = {}
    implied_outputs = []
    
    for r_id, antecedents, consequent in rules:
        alpha = min(mu[var][term] for var, term in antecedents)
        rule_activations[r_id] = alpha
        implied_mf = np.fmin(alpha, anomaly_mfs[consequent])
        implied_outputs.append(implied_mf)
        
    aggregated_mf = np.zeros_like(anomaly_universe)
    for mf in implied_outputs:
        aggregated_mf = np.fmax(aggregated_mf, mf)
        
    if np.sum(aggregated_mf) == 0:
        score = 0.0
    else:
        score = fuzz.defuzz(anomaly_universe, aggregated_mf, "centroid")
        
    if return_details:
        return score, rule_activations, mu, aggregated_mf
    return score

# 4. Tái tạo 3 kịch bản stream từ Phase 4
# Kịch bản 1: Control Stream
control_stream = stream_df.copy()

# Kịch bản 2: Sudden Drift Stream
SUDDEN_DRIFT_POINT = 1000
RPM_SHIFT = -150
TORQUE_SHIFT = 8
RPM_MIN = universes["rpm"].min()
RPM_MAX = universes["rpm"].max()
TORQUE_MIN = universes["torque"].min()
TORQUE_MAX = universes["torque"].max()

sudden_drift_stream = stream_df.copy()
after_drift = sudden_drift_stream.index >= SUDDEN_DRIFT_POINT
sudden_drift_stream.loc[after_drift, "Rotational speed [rpm]"] = (
    sudden_drift_stream.loc[after_drift, "Rotational speed [rpm]"] + RPM_SHIFT
).clip(RPM_MIN, RPM_MAX)
sudden_drift_stream.loc[after_drift, "Torque [Nm]"] = (
    sudden_drift_stream.loc[after_drift, "Torque [Nm]"] + TORQUE_SHIFT
).clip(TORQUE_MIN, TORQUE_MAX)

# Kịch bản 3: Gradual Drift Stream
GRADUAL_DRIFT_START = 800
GRADUAL_DRIFT_END = 1200
gradual_drift_stream = stream_df.copy()
gradual_drift_stream['Rotational speed [rpm]'] = gradual_drift_stream['Rotational speed [rpm]'].astype(float)
gradual_drift_stream['Torque [Nm]'] = gradual_drift_stream['Torque [Nm]'].astype(float)
for i in gradual_drift_stream.index:
    if i < GRADUAL_DRIFT_START:
        alpha = 0.0
    elif i >= GRADUAL_DRIFT_END:
        alpha = 1.0
    else:
        alpha = (i - GRADUAL_DRIFT_START) / (GRADUAL_DRIFT_END - GRADUAL_DRIFT_START)

    gradual_drift_stream.loc[i, "Rotational speed [rpm]"] = np.clip(
        stream_df.loc[i, "Rotational speed [rpm]"] + alpha * RPM_SHIFT,
        RPM_MIN, RPM_MAX
    )
    gradual_drift_stream.loc[i, "Torque [Nm]"] = np.clip(
        stream_df.loc[i, "Torque [Nm]"] + alpha * TORQUE_SHIFT,
        TORQUE_MIN, TORQUE_MAX
    )

scenario_streams = {
    "Control": control_stream,
    "Sudden Drift": sudden_drift_stream,
    "Gradual Drift": gradual_drift_stream,
}

print("Setup Phase 5 hoàn tất. Cả 3 kịch bản stream và Static Mamdani FIS Engine đã sẵn sàng.")


Setup Phase 5 hoàn tất. Cả 3 kịch bản stream và Static Mamdani FIS Engine đã sẵn sàng.


## Phase 5.1 — Chuẩn bị Anomaly-Score Stream

Chạy Static Fuzzy đã freeze ở Phase 3 trên cả ba kịch bản (`Control`, `Sudden Drift`, `Gradual Drift`).
Anomaly Score $A(t) \in [0, 1]$ được tính toán độc lập theo từng luồng, không sử dụng nhãn `Machine failure`.


In [2]:
# PHASE 5.1.1 — Prepare static fuzzy scores for all scenarios

def score_stream(df):
    return np.array([
        mamdani_inference(row)
        for _, row in df.iterrows()
    ])


scenario_scores = {}

for name, df in scenario_streams.items():

    print(f"Scoring {name}...")

    scenario_scores[name] = score_stream(df)

print("\nStatic Fuzzy scoring completed")
print("=============================")

for name, scores in scenario_scores.items():

    print(f"\n{name}")
    print("-" * len(name))

    print("Number of scores:", len(scores))
    print("Min:", scores.min())
    print("Max:", scores.max())
    print("Mean:", scores.mean())


Scoring Control...
Scoring Sudden Drift...
Scoring Gradual Drift...

Static Fuzzy scoring completed

Control
-------
Number of scores: 2000
Min: 0.0
Max: 0.6898147106136236
Mean: 0.31516101992410706

Sudden Drift
------------
Number of scores: 2000
Min: 0.19444479716810695
Max: 0.6898147106136236
Mean: 0.38198180231972056

Gradual Drift
-------------
Number of scores: 2000
Min: 0.19444479716810695
Max: 0.6898147106136236
Mean: 0.3824254413406079


## Phase 5.2 — Chọn Drift Detector (ADWIN)

Sử dụng thuật toán **ADWIN (Adaptive Windowing)** làm detector chính để giám sát luồng điểm bất thường $A(t) \in [0, 1]$.


In [1]:
# PHASE 5.2.0 — Verify Phase 5 kernel

import sys
import numpy as np
import river

from river.drift import ADWIN

print("Python:", sys.version)
print("Executable:", sys.executable)
print("NumPy:", np.__version__)
print("River:", river.__version__)

detector = ADWIN()

print("ADWIN:", detector)


Python: 3.11.16 | packaged by Anaconda, Inc. | (main, Aug 27 2026, 14:36:16) [MSC v.1942 64 bit (AMD64)]
Executable: C:\Users\vinhv\anaconda3\envs\drift_env\python.exe
NumPy: 2.4.6
River: 0.26.1
ADWIN: ADWIN


In [2]:
# PHASE 5.2.1 — ADWIN Toy Test

import numpy as np
from river.drift import ADWIN

np.random.seed(42)

# Synthetic score stream
before_drift = np.random.normal(
    loc=0.20,
    scale=0.02,
    size=500
)

after_drift = np.random.normal(
    loc=0.60,
    scale=0.02,
    size=500
)

toy_stream = np.concatenate([
    before_drift,
    after_drift
])

# Keep scores inside [0, 1]
toy_stream = np.clip(toy_stream, 0.0, 1.0)

# ADWIN
detector = ADWIN()

detections = []

for i, score in enumerate(toy_stream):
    detector.update(score)

    if detector.drift_detected:
        detections.append(i)

print("Toy stream length:", len(toy_stream))
print("Expected drift region: around index 500")
print("Detected drift indices:", detections)


Toy stream length: 1000
Expected drift region: around index 500
Detected drift indices: [543]


## Phase 5.3 — ADWIN trên 3 Kịch bản Stream

Áp dụng ADWIN trên luồng Anomaly Score $A(t) \in [0, 1]$ của từng kịch bản.
ADWIN chỉ quan sát giá trị $A(t)$ trực tuyến (unsupervised), hoàn toàn không nhận nhãn lỗi hay ground truth về trôi dạt.


In [3]:
# PHASE 5.3.1 — ADWIN on Control Stream

from river.drift import ADWIN

control_scores = scenario_scores["Control"]

control_detector = ADWIN()
control_detections = []

for i, score in enumerate(control_scores):
    control_detector.update(float(score))

    if control_detector.drift_detected:
        control_detections.append(i)

print("Scenario: Control")
print("=" * 40)
print("Stream length:", len(control_scores))
print("Detected drift indices:", control_detections)
print("Number of detections:", len(control_detections))


Scenario: Control
Stream length: 2000
Detected drift indices: []
Number of detections: 0


## Phase 5.4 — ADWIN trên Kịch bản Sudden Drift

Đánh giá năng lực phát hiện của ADWIN trên kịch bản trôi dạt đột ngột (**Sudden Drift**).
- Mốc tiêm trôi dạt ground truth: `index = 1000` (dịch chuyển tức thời RPM $-150\text{ rpm}$ và Torque $+8\text{ Nm}$).
- ADWIN chỉ quan sát luồng điểm số $A(t) \in [0, 1]$ độc lập và trực tuyến.


In [4]:
# PHASE 5.4.1 — ADWIN on Sudden Drift Stream

from river.drift import ADWIN

sudden_scores = scenario_scores["Sudden Drift"]

sudden_detector = ADWIN()
sudden_detections = []

for i, score in enumerate(sudden_scores):
    sudden_detector.update(float(score))

    if sudden_detector.drift_detected:
        sudden_detections.append(i)

print("Scenario: Sudden Drift")
print("=" * 40)
print("Stream length:", len(sudden_scores))
print("True injected drift index:", 1000)
print("Detected drift indices:", sudden_detections)
print("Number of detections:", len(sudden_detections))


Scenario: Sudden Drift
Stream length: 2000
True injected drift index: 1000
Detected drift indices: [1183]
Number of detections: 1


## Phase 5.5 — ADWIN trên Kịch bản Gradual Drift

Đánh giá hành vi phát hiện của ADWIN trên kịch bản trôi dạt tiệm tiến (**Gradual Drift**).
- Vùng chuyển tiếp ground truth: $t = 800 \to 1200$ (chuyển tiếp tuyến tính 400 mẫu, mốc giữa $t=1000$).
- ADWIN giám sát trực tuyến luồng $A(t) \in [0, 1]$ độc lập, không nhận nhãn lỗi hay thông tin vùng chuyển tiếp.


In [5]:
# PHASE 5.5.1 — ADWIN on Gradual Drift Stream

from river.drift import ADWIN

gradual_scores = scenario_scores["Gradual Drift"]

gradual_detector = ADWIN()
gradual_detections = []

for i, score in enumerate(gradual_scores):
    gradual_detector.update(float(score))

    if gradual_detector.drift_detected:
        gradual_detections.append(i)

print("Scenario: Gradual Drift")
print("=" * 40)
print("Stream length:", len(gradual_scores))
print("Transition start:", 800)
print("Transition end:", 1200)
print("Detected drift indices:", gradual_detections)
print("Number of detections:", len(gradual_detections))


Scenario: Gradual Drift
Stream length: 2000
Transition start: 800
Transition end: 1200
Detected drift indices: [1247]
Number of detections: 1


## Phase 5.6 — Tổng hợp Kết quả Drift Detector

Tổng hợp kết quả phát hiện trôi dạt của ADWIN trên toàn bộ 3 kịch bản thực nghiệm đối chuẩn (`Control`, `Sudden Drift`, `Gradual Drift`).


In [6]:
# PHASE 5.6.1 — Drift Detection Summary

drift_summary = [
    {
        "Scenario": "Control",
        "Drift Region": "None",
        "Detections": control_detections,
        "Number of Detections": len(control_detections),
    },
    {
        "Scenario": "Sudden Drift",
        "Drift Region": "Index 1000",
        "Detections": sudden_detections,
        "Number of Detections": len(sudden_detections),
    },
    {
        "Scenario": "Gradual Drift",
        "Drift Region": "Index 800-1200",
        "Detections": gradual_detections,
        "Number of Detections": len(gradual_detections),
    },
]

for result in drift_summary:
    print(result)


{'Scenario': 'Control', 'Drift Region': 'None', 'Detections': [], 'Number of Detections': 0}
{'Scenario': 'Sudden Drift', 'Drift Region': 'Index 1000', 'Detections': [1183], 'Number of Detections': 1}
{'Scenario': 'Gradual Drift', 'Drift Region': 'Index 800-1200', 'Detections': [1247], 'Number of Detections': 1}
